In [ ]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pathlib import Path

SEED = 42
DECAY_GAMMA = 0.03
EB_KAPPA = 500

# Paths
TX_PATH = "../data/curated/merchant_transactions"
PROB_PATH = "../data/tables/merchant_data/consumer_fraud_probability.csv"
OUTPUT_DIR = "../artifacts/fraud_outputs"
MERCHANTS_PATH = "../data/tables/merchant_data/tbl_merchants.parquet"
CURATED_DIR = "../data/curated"

conf = (
    SparkConf()
    .setAppName("fraud_prob_pipeline")
    .set("spark.sql.adaptive.enabled", "true")
    .set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .set("spark.sql.shuffle.partitions", "200")
    .set("spark.driver.memory", "6g")
    .set("spark.executor.memory", "6g")
)

spark = (
    SparkSession.builder.master("local[*]").config(conf=conf).getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

### Pipeline overview
- Load transactions and direct fraud probabilities
- Clean transactions
- Tiered probability: direct match (Tier A), user-decay imputation (Tier B), model-based imputation (Tier C)
- Aggregate to merchant-level with empirical Bayes shrinkage and compute FraudScore
- Report merchant rankings


In [ ]:
# Global tunables
LAMBDA = 1.0  # EB/fraud score adjustment strength


### Load & Basic Hygiene


In [ ]:
# Read inputs

# Define schemas for strict typing
schema_tx = T.StructType([
    T.StructField("merchant_abn", T.LongType(), True),
    T.StructField("user_id", T.LongType(), True),
    T.StructField("dollar_value", T.DoubleType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("business", T.StringType(), True),
    T.StructField("biz_tags", T.StringType(), True),
    T.StructField("rev_band", T.StringType(), True),
    T.StructField("take_rate", T.StringType(), True),
])

schema_prob = T.StructType([
    T.StructField("user_id", T.LongType(), True),
    T.StructField("order_datetime", T.DateType(), True),
    T.StructField("fraud_probability", T.DoubleType(), True),
])

# Load primary inputs
transactions = spark.read.schema(schema_tx).parquet(TX_PATH)
probs = spark.read.option("header", True).schema(schema_prob).csv(PROB_PATH)

# Normalize probabilities to [0,1] if given as percent
probs = probs.withColumn(
    "fraud_probability",
    F.when(F.col("fraud_probability") > 1.0, F.col("fraud_probability") / F.lit(100.0)).otherwise(F.col("fraud_probability"))
)

# Text hygiene & parse merchant take_rate to numeric
transactions = (
    transactions
    .withColumn("biz_tags", F.trim(F.regexp_replace(F.col("biz_tags"), "\s+", " ")))
    .withColumn("rev_band", F.trim(F.col("rev_band")))
    .withColumn("take_rate", F.trim(F.col("take_rate")))
    .withColumn(
        "take_rate_num",
        F.when(F.col("take_rate").rlike(r"^[0-9.]+%$"), F.regexp_replace("take_rate", "%", "").cast("double") / 100.0)
         .when(F.col("take_rate").rlike(r"^[0-9.]+$"), F.col("take_rate").cast("double"))
         .otherwise(F.lit(None).cast("double"))
    )
)

# Cache & counts
transactions.cache(); probs.cache()
print("Transactions:", transactions.count())
print("Probs rows:", probs.count())


In [ ]:
transactions.show(20)
probs.show(20)

### Tier A: Direct probability matches


In [ ]:
# Left join on (user_id, order_datetime) to get p_direct

transactions = transactions.withColumn("order_date", F.col("order_datetime"))
probs = probs.withColumn("order_date", F.col("order_datetime")).drop("order_datetime")

joined_a = transactions.join(
    probs.select("user_id", "order_date", F.col("fraud_probability").alias("p_direct")),
    ["user_id", "order_date"],
    "left",
)

print("Tier A: p_direct non-null:", joined_a.filter(F.col("p_direct").isNotNull()).count())
joined_a.cache()


### Tier B: User-level propensity with time decay


In [ ]:
# Compute p_user_decay for rows without p_direct

# Prepare per-user probability history as (user_id, d_k, p_k)
prob_hist = probs.select(
    "user_id", F.col("order_date").alias("d_k"), F.col("fraud_probability").alias("p_k")
)

# For efficiency: join only for users present in transactions lacking p_direct
users_needing = joined_a.filter(F.col("p_direct").isNull()).select("user_id").distinct()

cand = (
    joined_a.select("user_id", "order_date")
    .join(users_needing, "user_id", "inner")
    .join(prob_hist, "user_id", "inner")
)

# Compute weights w_k = exp(-gamma * |t - d_k|) in days
diff_days = F.abs(F.datediff(F.col("order_date"), F.col("d_k")))
cand = cand.withColumn("w_k", F.exp(-DECAY_GAMMA * diff_days))

# Aggregate per (user_id, order_date)
p_user_decay_df = (
    cand.groupBy("user_id", "order_date")
    .agg(
        (F.sum(F.col("p_k") * F.col("w_k")) / F.sum(F.col("w_k"))).alias("p_user_decay")
    )
)

# Join back onto joined_a, only where p_direct is null
joined_b = (
    joined_a
    .join(p_user_decay_df, ["user_id", "order_date"], "left")
    .withColumn("p_user_decay", F.when(F.col("p_direct").isNull(), F.col("p_user_decay")).otherwise(F.lit(None)))
)

print("Tier B: filled via decay:", joined_b.filter(F.col("p_user_decay").isNotNull()).count())
joined_b.cache()


In [ ]:
joined_b.show(20)

### Tier C: Model to impute remaining probabilities (soft-label regression)


### Tier C Features


In [ ]:
# Pre-training feature significance checks
# - Pearson correlation vs p_direct for numeric features

from pyspark.sql import functions as F

# Build minimal feature base if not present
if 'base' not in locals():
    if 'joined_b' in locals():
        ds = joined_b
    elif 'joined_a' in locals():
        ds = joined_a
    else:
        raise ValueError("Run Tier A/B cells first to create joined_a/joined_b before significance checks.")
    base = ds.withColumn("log_amount", F.log1p(F.col("dollar_value"))) \
             .withColumn("dow", F.dayofweek("order_date")) \
             .withColumn("month", F.month("order_date"))
    w_user_90 = (
        Window.partitionBy("user_id").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
    )
    base = base.withColumn("user_txn_count_90d", F.count(F.lit(1)).over(w_user_90)) \
               .withColumn("user_sum_90d", F.sum("dollar_value").over(w_user_90)) \
               .withColumn("user_avg_amount_90d", F.avg("dollar_value").over(w_user_90))
    w_user_prev = Window.partitionBy("user_id").orderBy("order_date")
    base = base.withColumn("prev_date", F.lag("order_date").over(w_user_prev)) \
               .withColumn("user_days_since_prev", F.datediff("order_date", "prev_date")) \
               .drop("prev_date")
    w_merch_90 = (
        Window.partitionBy("merchant_abn").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
    )
    base = base.withColumn("m_txn_count_90d", F.count(F.lit(1)).over(w_merch_90)) \
               .withColumn("m_sum_90d", F.sum("dollar_value").over(w_merch_90)) \
               .withColumn("m_avg_amount_90d", F.avg("dollar_value").over(w_merch_90))
    numeric_fill = [
        "log_amount", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
        "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
    ]
    base = base.fillna(0, subset=numeric_fill)

# Use labeled rows (where p_direct is available)
if 'labeled' not in locals():
    labeled = base.filter(F.col("p_direct").isNotNull()).cache()

print("Labeled rows for significance checks:", labeled.count())

numeric_features = [
    "log_amount",
    "dow",
    "month",
    "user_txn_count_90d",
    "user_sum_90d",
    "user_avg_amount_90d",
    "user_days_since_prev",
    "m_txn_count_90d",
    "m_sum_90d",
    "m_avg_amount_90d",
    "take_rate_num",
]

# Pearson correlations
corr_results = []
for feat in numeric_features:
    try:
        c = labeled.stat.corr(feat, "p_direct")
    except Exception:
        c = None
    corr_results.append((feat, c))

# Print correlations sorted by absolute value
corr_results_sorted = sorted(
    [(f, c) for f, c in corr_results if c is not None], key=lambda x: abs(x[1]), reverse=True
)
print("Pearson correlation with p_direct (top):")
for f, c in corr_results_sorted:
    print(f"  {f}: {c:.6f}")


In [ ]:
# Feature engineering for modeling (no income)

# Base for modeling
base = joined_b.withColumn(
    "p_label", F.coalesce(F.col("p_direct"), F.lit(None))
)

# Transaction-level
base = base.withColumn("log_amount", F.log1p(F.col("dollar_value"))) \
           .withColumn("dow", F.dayofweek("order_date")) \
           .withColumn("month", F.month("order_date"))

# User windows (90d)
w_user_90 = (
    Window.partitionBy("user_id").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
)
base = base.withColumn("user_txn_count_90d", F.count(F.lit(1)).over(w_user_90)) \
           .withColumn("user_sum_90d", F.sum("dollar_value").over(w_user_90)) \
           .withColumn("user_avg_amount_90d", F.avg("dollar_value").over(w_user_90))

# Days since previous txn per user
w_user_prev = Window.partitionBy("user_id").orderBy("order_date")
base = base.withColumn("prev_date", F.lag("order_date").over(w_user_prev)) \
           .withColumn("user_days_since_prev", F.datediff("order_date", "prev_date")) \
           .drop("prev_date")

# Merchant windows (90d)
w_merch_90 = (
    Window.partitionBy("merchant_abn").orderBy(F.col("order_date").cast("timestamp").cast("long")).rangeBetween(-90*86400, 0)
)
base = base.withColumn("m_txn_count_90d", F.count(F.lit(1)).over(w_merch_90)) \
           .withColumn("m_sum_90d", F.sum("dollar_value").over(w_merch_90)) \
           .withColumn("m_avg_amount_90d", F.avg("dollar_value").over(w_merch_90))

# Impute missing numeric features to avoid NaN/Inf in vectors
numeric_fill = [
    "log_amount", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]
base = base.fillna(0, subset=numeric_fill)

# Categorical encodings: biz_tags, rev_band
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

indexers = [
    StringIndexer(inputCol="rev_band", outputCol="rev_band_idx", handleInvalid="keep"),
    StringIndexer(inputCol="biz_tags", outputCol="biz_tags_idx", handleInvalid="keep"),
]
encoders = [
    OneHotEncoder(inputCols=["rev_band_idx", "biz_tags_idx"], outputCols=["rev_band_oh", "biz_tags_oh"])
]

feats = [
    "log_amount", "dow", "month", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]

assembler = VectorAssembler(inputCols=feats + ["rev_band_oh", "biz_tags_oh"], outputCol="features", handleInvalid="keep")

prep_pipeline = Pipeline(stages=indexers + encoders + [assembler])

# Labeled training data: where p_direct is available
labeled = base.filter(F.col("p_direct").isNotNull()).cache()
print("Labeled rows:", labeled.count())


In [ ]:
# Ablation study: retrain without income features (same split/seed)
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# Prepare ablation dataset by removing income features
indexers_no_income = [
    StringIndexer(inputCol="rev_band", outputCol="rev_band_idx", handleInvalid="keep"),
    StringIndexer(inputCol="biz_tags", outputCol="biz_tags_idx", handleInvalid="keep"),
]
encoders_no_income = [
    OneHotEncoder(inputCols=["rev_band_idx", "biz_tags_idx"], outputCols=["rev_band_oh", "biz_tags_oh"])
]

feats_no_income = [
    "log_amount", "dow", "month", "user_txn_count_90d", "user_sum_90d", "user_avg_amount_90d",
    "user_days_since_prev", "m_txn_count_90d", "m_sum_90d", "m_avg_amount_90d", "take_rate_num"
]

assembler_no_income = VectorAssembler(inputCols=feats_no_income + ["rev_band_oh", "biz_tags_oh"], outputCol="features", handleInvalid="keep")

prep_pipeline_no_income = Pipeline(stages=indexers_no_income + encoders_no_income + [assembler_no_income])

prepared_no_income = prep_pipeline_no_income.fit(labeled).transform(labeled)
train_df_no_income, valid_df_no_income = prepared_no_income.randomSplit([0.8, 0.2], seed=SEED)

# Define models
lr_no_income = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt_no_income = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

# Train
lr_model_no_income = lr_no_income.fit(train_df_no_income)
gbt_model_no_income = gbt_no_income.fit(train_df_no_income)

# Predict
pred_lr_no_income = lr_model_no_income.transform(valid_df_no_income)
pred_gbt_no_income = gbt_model_no_income.transform(valid_df_no_income)

# Clip to [0,1]
def _clip01_df(df, labelCol="p_direct", predCol="prediction"):
    return df.withColumn("lbl", F.when(F.col(labelCol) < 0, 0.0).when(F.col(labelCol) > 1, 1.0).otherwise(F.col(labelCol))) \
             .withColumn("prd", F.when(F.col(predCol) < 0, 0.0).when(F.col(predCol) > 1, 1.0).otherwise(F.col(predCol)))

mae_eval = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mse")

cl_lr = _clip01_df(pred_lr_no_income)
cl_gbt = _clip01_df(pred_gbt_no_income)

metrics_ablate = {
    "lr_mae": mae_eval.evaluate(cl_lr),
    "lr_brier": mse_eval.evaluate(cl_lr),
    "gbt_mae": mae_eval.evaluate(cl_gbt),
    "gbt_brier": mse_eval.evaluate(cl_gbt),
}
print("Ablation metrics (no income):", metrics_ablate)

# If ablation is worse than original, try simpler GBT / stronger LR
try:
    baseline_metrics = metrics  # from previous training cell
except NameError:
    baseline_metrics = None

if baseline_metrics is None or (metrics_ablate["gbt_mae"] > baseline_metrics.get("gbt_mae", 1e9)):
    print("Ablation worse than baseline; simplifying models...")
    lr_tuned = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.7, regParam=0.3)
    gbt_tuned = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=4, maxIter=40, stepSize=0.08, subsamplingRate=0.8)

    lr_model_tuned = lr_tuned.fit(train_df_no_income)
    gbt_model_tuned = gbt_tuned.fit(train_df_no_income)

    pred_lr_tuned = lr_model_tuned.transform(valid_df_no_income)
    pred_gbt_tuned = gbt_model_tuned.transform(valid_df_no_income)

    cl_lr_tuned = _clip01_df(pred_lr_tuned)
    cl_gbt_tuned = _clip01_df(pred_gbt_tuned)

    metrics_tuned = {
        "lr_mae": mae_eval.evaluate(cl_lr_tuned),
        "lr_brier": mse_eval.evaluate(cl_lr_tuned),
        "gbt_mae": mae_eval.evaluate(cl_gbt_tuned),
        "gbt_brier": mse_eval.evaluate(cl_gbt_tuned),
    }
    print("Tuned metrics (no income):", metrics_tuned)



In [ ]:
# Train LR and GBT; evaluate and calibrate

from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

prepared = prep_pipeline.fit(labeled).transform(labeled)

train_df, valid_df = prepared.randomSplit([0.8, 0.2], seed=SEED)

lr = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

lr_model = lr.fit(train_df)
gbt_model = gbt.fit(train_df)

pred_lr = lr_model.transform(valid_df)
pred_gbt = gbt_model.transform(valid_df)

# Evaluate MAE and Brier (MSE on [0,1])
mae_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mse")

def _clip01(col):
    return F.when(F.col(col) < 0, 0.0).when(F.col(col) > 1, 1.0).otherwise(F.col(col))

def evaluate(df, label="p_direct", pred="prediction"):
    tmp = df.withColumn("lbl", _clip01(label)).withColumn("prd", _clip01(pred))
    mae   = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mae").evaluate(tmp)
    brier = RegressionEvaluator(labelCol="lbl", predictionCol="prd", metricName="mse").evaluate(tmp)
    return mae, brier

metrics = {
    "lr_mae": evaluate(pred_lr)[0],
    "lr_brier": evaluate(pred_lr)[1],
    "gbt_mae": evaluate(pred_gbt)[0],
    "gbt_brier": evaluate(pred_gbt)[1],
}
print(metrics)

# Choose best model (skip isotonic calibration for speed)
best_is_gbt = metrics["gbt_mae"] <= metrics["lr_mae"]
best_model = gbt_model if best_is_gbt else lr_model


In [ ]:
# Apply best model to unlabeled rows and calibrate

prep_model = prep_pipeline.fit(labeled)
prepared = prep_model.transform(labeled)

# Re-train both on prepared
train_df, valid_df = prepared.randomSplit([0.8, 0.2], seed=SEED)

lr = LinearRegression(labelCol="p_direct", featuresCol="features", elasticNetParam=0.5, regParam=0.1)
gbt = GBTRegressor(labelCol="p_direct", featuresCol="features", maxDepth=6, maxIter=60, stepSize=0.1, subsamplingRate=0.8)

lr_model = lr.fit(train_df)
gbt_model = gbt.fit(train_df)

pred_lr = lr_model.transform(valid_df)
pred_gbt = gbt_model.transform(valid_df)

mae_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mae")
mse_eval = RegressionEvaluator(labelCol="p_direct", predictionCol="prediction", metricName="mse")

metrics = {
    "lr_mae": mae_eval.evaluate(pred_lr),
    "lr_brier": mse_eval.evaluate(pred_lr),
    "gbt_mae": mae_eval.evaluate(pred_gbt),
    "gbt_brier": mse_eval.evaluate(pred_gbt),
}
print("Validation metrics:", metrics)

best_is_gbt = metrics["gbt_mae"] <= metrics["lr_mae"]
best_model = gbt_model if best_is_gbt else lr_model
best_valid = pred_gbt if best_is_gbt else pred_lr

unlabeled = base.filter(F.col("p_direct").isNull() & F.col("p_user_decay").isNull())
prepared_unlabeled = prep_model.transform(unlabeled)

scored_unlabeled = best_model.transform(prepared_unlabeled)

# Clip predictions to [0,1] as calibrated scores
scored_unlabeled = scored_unlabeled.withColumn(
    "p_model_calibrated",
    F.when(F.col("prediction") < 0, 0.0).when(F.col("prediction") > 1, 1.0).otherwise(F.col("prediction"))
)

print("Scored unlabeled rows:", scored_unlabeled.count())


In [ ]:
# Coalesce probabilities and persist per-transaction output

# Bring together tiers
coalesced = (
    joined_b
    .join(scored_unlabeled.select("merchant_abn", "user_id", "order_date", "p_model_calibrated"), ["merchant_abn", "user_id", "order_date"], "left")
    .withColumn("p_hat", F.coalesce(F.col("p_direct"), F.col("p_user_decay"), F.col("p_model_calibrated")))
)

# Clip to [0,1], fallback to global mean if any remain null
mu = coalesced.select(F.mean("p_hat")).first()[0]
coalesced = coalesced.withColumn("p_hat", F.when(F.col("p_hat").isNull(), F.lit(mu)).otherwise(F.col("p_hat")))
coalesced = coalesced.withColumn("p_hat", F.when(F.col("p_hat") < 0, 0.0).when(F.col("p_hat") > 1, 1.0).otherwise(F.col("p_hat")))

per_tx_out = coalesced.select(
    "merchant_abn", "user_id", F.col("order_date").alias("order_datetime"), "dollar_value", "biz_tags", "rev_band", "take_rate", "p_hat"
)

print("Per-transaction output rows:", per_tx_out.count())


### Merchant-level aggregation & Empirical-Bayes shrinkage


In [ ]:
# Aggregate to merchant metrics and EB shrinkage

agg = (
    per_tx_out.groupBy("merchant_abn")
    .agg(
        F.count(F.lit(1)).alias("n_txn"),
        F.sum("dollar_value").alias("sum_amount"),
        F.avg("p_hat").alias("mean_p"),
        F.sum(F.col("p_hat") * F.col("dollar_value")).alias("EFL"),
    )
    .withColumn("EFLR", F.col("EFL") / F.when(F.col("sum_amount") == 0, F.lit(1.0)).otherwise(F.col("sum_amount")))
)

mu = per_tx_out.select(F.avg("p_hat").alias("mu")).first()[0]

agg = agg.withColumn(
    "eb_p",
    (F.col("n_txn") * F.col("mean_p") + F.lit(EB_KAPPA) * F.lit(mu)) / (F.col("n_txn") + F.lit(EB_KAPPA))
)

agg = agg.withColumn(
    "se_mean_p",
    F.sqrt(F.col("mean_p") * (1 - F.col("mean_p")) / F.greatest(F.col("n_txn"), F.lit(1)))
)


In [ ]:
# Rankings: Best-100 (safest to onboard) and Worst-100 (investigate)


merchants_sel = (
    spark.read.parquet(MERCHANTS_PATH)
    .select("merchant_abn", F.col("name").alias("merchant_name"))
)

agg_named = agg.join(F.broadcast(merchants_sel), "merchant_abn", "left")

best_100 = (
    agg_named
    .orderBy(F.col("eb_p").asc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(100)
)
worst_100 = (
    agg_named
    .orderBy(F.col("EFL").desc())
    .select("merchant_abn", "merchant_name", "n_txn", "sum_amount", "eb_p", "EFL", "EFLR")
    .limit(100)
)

print("Best-100 (safest to onboard):")
best_100.show(20, truncate=False)
print("Worst-100 (investigate):")
worst_100.show(20, truncate=False)

# Save to curated folder
best_100.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/best_100_merchants.csv")
worst_100.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/worst_100_merchants.csv")

print("Saved rankings to curated folder")

In [ ]:
agg_named.write.mode("overwrite").option("header", True).csv(f"{CURATED_DIR}/merchant_fraud_rankings.csv")